# Lab 5: Classifying user profiles

Let's say you're a developer working at a social media company, that makes money by selling targeted advertisements. In order to do this, you need to classify your platform's users into specific personas! 

To do this, let's try and figure out the top 5 topics a particular Bluesky user is interested in!

First, let's log in to Bluesky as usual.

In [ ]:
# Login to Bluesky
from atproto import Client
%run bluesky_keys.py

client = Client(base_url="https://bsky.social")
client.login(handle, password)

Now, your job is to create a function that accepts a Bluesky user's handle, and returns their top 5 "topics" (these can just be keywords). You can do this by using a dictionary counter to keep track of every unique word in a post, counting how many times they appear, and putting the top 5 into a ranked list.

To do that, it will be helpful to be able to keep track of individual words! You can use the split function to do this easily, and don't forget to make sure everything is lowercase:

In [ ]:
"Let's split up a string!".lower().split()

For this lab, I'll use my own Bluesky profile as an example! The code below uses my handle to get my most recent 100 posts to Bluesky. If you have an account of your own, you can feel free to use that instead, but let's not use any random person's account for now; as we discussed in class, the ethics around inferring these things from social media can be contentious!

The result of the code below is a list of strings, each representing the text of my (or your, if you change the handle) posts. 

In [ ]:
profile = client.get_profile(actor="swilliams1918.bsky.social")

next_page = None
all_posts = []
while True:
    data = client.get_author_feed(
        actor=profile['did'],
        filter='posts_and_author_threads',
        limit=100,
        cursor=next_page
    )
    feed = data.feed
    for fvp in feed:
        all_posts.append(fvp.post.record.text)
    next_page = data.cursor
    if not next_page:
        break

If you want, you can print the list of posts (but note, it'll be pretty long)

In [ ]:
print(all_posts)

Now, it's time to define a function that takes this list, and generates the top 5 keywords associated with this user!

Right now, it accepts a list of posts, creates an empty list of topics, which it returns as output. Your job is to fill in the rest, and test it!

Hint: You'll want to use a dictionary counter for this; feel free to return to lecture 9 practice for examples!

In [ ]:
def top_keywords(list_of_posts):
    topics = []
    #TODO: Write code to add the top 5 most common keywords in list_of_posts to the topics list
    return topics

#test your function here!

After you get it working, you might notice some issues. For example, words like "the", "and", "it", etc aren't really topics, but they probably keep appearing as your top topics! How can we get rid of them?

For that, we can turn to information theory! There's a term called "inverse-document frequency," that basically tries to quantify how much information a word provides (i.e. how rare it is). Basically, we divide the number of total number of documents, by the number of documents a word appears in. 

For example, if we have three sentences:
1. "The best ice cream flavor is chocolate."
2. "Only idiots think the best ice cream flavor is chocolate."
3. "I hate ice cream"

...the word "the" appears in two documents (1 and 2), and there are three documents total. So, the idf for the word "the" is 3/2 = 1.5. 

To try and get better topics, your next step is to define a function to calculate the idf (inverse-document frequency) of a word. This will later help you to decide whether some words are too common to consider "topics"! 

First, I asked ChatGPT to come up with a random set of hundreds of made-up Reddit titles and Bluesky posts to serve as a corpus of documents:

In [ ]:
%run post_corpus.ipynb

Now, here is a function that uses that corpus to calculate an idf score for a given word. Note that I'm adding a constant +1 to the score, just in case the term never appears in the corpus at all (thus avoiding a division by 0 error). 

In [39]:
def idf(term):
    term = term.lower()
    docs_appeared = 1
    for p in corpus:
        words = p.lower().split()
        if term in words:
            docs_appeared += 1
    return len(corpus) / docs_appeared

#To give you a sense of the range of these scores...
print("Probably not meaningful:")
print(idf("the"))
print(idf("a"))
print("\n")

print("Probably meaningful:")
print(idf("AI"))
print(idf("movie"))
print("\n")

print("Maximum idf score:")
print(idf("abcdefgh"))

Probably not meaningful:
8.573333333333334
3.2974358974358973


Probably meaningful:
53.583333333333336
160.75


Maximum idf score:
643.0


Your last task: Copy-paste your function from above, and try to use the idf score of common terms to create a more meaningful list of keywords. You can try using an idf cut-off, multiplying the term frequency with its idf score, etc. Does the idf score help you?